# Smoke test — Qwen2-Audio-7B-Instruct on the pilot stimuli

Stage 0 of `report/Task.md`: before running the full pilot, confirm on a few samples that

1. the model loads and runs on audio + text inputs,
2. the **logit readout works**: almost all next-token probability lands on `A` or `B` (`mass_ab` ≈ 1), and
3. it **agrees with what the model actually says** in free-form generation (`greedy_agrees` ≈ 1), and
4. the **Yes/No readout** works too: almost all probability lands on `Yes` or `No` (`mass_yn` ≈ 1).

If (2) or (3) fails, switch the pilot to `uq: self_consistency`.

**Before running:** start Jupyter from the repo (any folder inside it works) in an environment with a CUDA GPU.
≥ 20 GB total VRAM runs fp16 (split across GPUs if needed); a single 16 GB GPU falls back to 4-bit.
The first run downloads the model (~15 GB) into the Hugging Face cache (`~/.cache/huggingface`, or `HF_HOME`).

The run covers 2 fictitious + 2 control facts × 9 conditions (conflict items in both A/B orders) = 44 stimuli.

## 1. Project root and GPU check

In [ ]:
import os, sys, torch
from pathlib import Path

# the kernel may start in notebooks/ -> move to the repo root so all relative paths work
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/eval/run_model.py").exists())
os.chdir(REPO)
print("working in", REPO)

# load KEY=VALUE pairs from the repo's .env (git-ignored), e.g. HF_TOKEN for the Hugging Face download;
# variables already set in the environment win. The run_model commands below inherit them.
env_file = REPO / ".env"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.removeprefix("export ").split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip("\"'"))
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set (fine for public models)")

assert torch.cuda.is_available(), "No GPU visible to this kernel (check CUDA_VISIBLE_DEVICES / the driver)"
gpus = [torch.cuda.get_device_properties(i) for i in range(torch.cuda.device_count())]
VRAM_GB = sum(g.total_memory for g in gpus) / 1e9
for g in gpus:
    print(f"{g.name}: {g.total_memory / 1e9:.1f} GB")
# 7B in fp16 needs ~16 GB of weights plus activations; device_map="auto" splits it across GPUs
QUANTIZATION = "none" if VRAM_GB >= 20 else "4bit"
# native bf16 needs compute capability >= 8.0 (Ampere or newer); older GPUs only emulate it -> float16
DTYPE = "bfloat16" if min(torch.cuda.get_device_capability(i)[0] for i in range(len(gpus))) >= 8 else "float16"
print(f"total {VRAM_GB:.1f} GB -> quantization: {QUANTIZATION} | dtype: {DTYPE}")

## 2. Install dependencies
Installs `requirements.txt` into the environment of **this kernel** (`sys.executable`). Skip it if the environment is already set up.

In [ ]:
INSTALL = False
if INSTALL:
    !{sys.executable} -m pip install -q -r requirements.txt
import transformers; print("transformers", transformers.__version__)

## 3. Local config
Copies `configs/pilot.yaml` to `configs/local.yaml` (git-ignored) with the quantization / dtype chosen for this GPU.

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/pilot.yaml"))
cfg["model"]["quantization"] = QUANTIZATION
cfg["model"]["dtype"] = DTYPE
yaml.safe_dump(cfg, open("configs/local.yaml", "w"), sort_keys=False)
print(yaml.safe_dump(cfg["model"]), "eval:", cfg["eval"]["uq"], "| greedy_check:", cfg["eval"]["greedy_check"])

## 4. Check the data
The stimuli and audio come from the repo; they are regenerated only if missing (TTS needs internet).

In [ ]:
import glob, json
if not os.path.exists("data/generated/stimuli.jsonl"):
    !{sys.executable} -m src.data_gen.text_variants
n_audio = len(glob.glob("data/audio/*.mp3"))
print("audio clips:", n_audio)
if n_audio == 0:
    !{sys.executable} -m src.data_gen.tts
    print("audio clips:", len(glob.glob("data/audio/*.mp3")))

In [ ]:
# listen to one clip to make sure the TTS is intelligible
from IPython.display import Audio, display
stim = json.loads(open("data/generated/stimuli.jsonl").readline())
print(stim["question"], "| audio says:", stim["audio_value"])
display(Audio(stim["audio"]))

## 5. Run the smoke test

`--limit 2` = the first 2 fictitious and 2 control facts, all conditions.
Re-running the cell resumes where it stopped; delete `results/smoke` to start fresh.

In [ ]:
!{sys.executable} -m src.eval.run_model --config configs/local.yaml --out results/smoke --limit 2

## 6. Read the result

In [ ]:
from IPython.display import Markdown
display(Markdown(open("results/smoke/summary.md").read()))

In [ ]:
import pandas as pd
rows = [json.loads(l) for l in open("results/smoke/predictions.jsonl")]
df = pd.DataFrame(rows)
cols = ["stim_id", "expected", "chosen_value", "confidence", "mass_ab", "top_token", "greedy_letter", "chosen_letter", "greedy_raw"]
pd.set_option("display.max_colwidth", 80)
df[[c for c in cols if c in df]]

### Verdict

- `mass_ab` ≥ 0.9 → the pre-filled prompt really makes the next token a letter.
- `greedy_agrees` ≥ 0.9 → the logit readout matches the model's own free-form answer.
- `greedy_format_ok` → how often the free-form answer is parseable JSON with a letter.
- `mass_yn` ≥ 0.9 → the Yes/No prompt really makes the next token Yes or No.

In [ ]:
mass = df["mass_ab"].mean() if "mass_ab" in df else float("nan")
has_greedy = "greedy_letter" in df
agree = (df["greedy_letter"] == df["chosen_letter"]).mean() if has_greedy else float("nan")
fmt_ok = df["greedy_letter"].notna().mean() if has_greedy else float("nan")
print(f"mass_ab          {mass:.3f}")
print(f"greedy_agrees    {agree:.3f}")
print(f"greedy_format_ok {fmt_ok:.3f}")
yn = [o["mass_yn"] for r in rows if r.get("yes_no_out") for o in r["yes_no_out"].values()]
mass_yn = sum(yn) / len(yn) if yn else float("nan")
print(f"mass_yn          {mass_yn:.3f}  ({len(yn)} Yes/No passes)")
if mass >= 0.9 and agree >= 0.9 and mass_yn >= 0.9:
    print("\nPASS: A/B and Yes/No readouts are reliable -> run the full pilot.")
else:
    print("\nCHECK: logit readout looks unreliable -> inspect top_token / greedy_raw above;"
          " if confirmed, run the pilot with --uq self_consistency (section 8).")

## 7. (Only if the verdict failed) self-consistency fallback

Samples 10 free-form answers per stimulus instead of reading logits. Slower (~10× generation).

In [ ]:
RUN_FALLBACK = False
if RUN_FALLBACK:
    !{sys.executable} -m src.eval.run_model --config configs/local.yaml --out results/smoke_sc --limit 2 --uq self_consistency
    display(Markdown(open("results/smoke_sc/summary.md").read()))

## 8. Results
Everything is written to `results/smoke/` in the repo (`predictions.jsonl`, `metrics.json`, `summary.md`, `config_used.json`).